# Chapter 2: Ensemble Learning & Random Forest

> **Chapter Goal:** Chapter 1 showed that single trees fail because of **high variance**. This chapter presents the first major solution: **Random Forest**. We build the theory from first principles — why averaging helps, why it has limits, how Random Forest pushes past those limits, and critically, **how to explain ensemble predictions using SHAP values**.

---

## 1. Ensemble Learning: The Mathematical Foundation

### **The Wisdom of Crowds — But Why Does It Work?**
The intuition is that "many heads are better than one." But this is only true under specific conditions. Let's prove it mathematically.

### **A. Variance Reduction Through Averaging (The Core Theorem)**
Let's say you have $N$ models, each with the same variance $\sigma^2$ and **zero correlation** with each other (fully independent predictions).

The variance of the **average** of $N$ independent predictions is:
$$Var\left(\frac{1}{N}\sum_{i=1}^{N} T_i\right) = \frac{\sigma^2}{N}$$

If you average 100 trees, each with variance $\sigma^2 = 1.0$, the ensemble's variance drops to **0.01**. This is a 100x reduction — the mathematical reason why ensembles work.

**The Bias stays the same:** Averaging unbiased estimators gives an unbiased estimator. We get variance reduction **for free**!

### **B. The Correlation Problem**
The theorem assumed **zero correlation** between trees. In practice, trees trained on the same dataset  pick the same dominant features and make similar errors.

$$Var(Ensemble) = \rho \cdot \sigma^2 + \frac{1-\rho}{N} \cdot \sigma^2$$

- $\rho \cdot \sigma^2$ — **Cannot be reduced** by adding more trees. This is the variance floor.
- $\frac{(1-\rho)}{N} \cdot \sigma^2$ — **Can be reduced** by adding more trees or reducing correlation.

**This is why Random Forest introduces randomness** — to actively reduce $\rho$.

### **C. The Two Ensemble Strategies Compared**
| | Bagging | Boosting |
| :--- | :--- | :--- |
| **Build order** | Parallel (all at once) | Sequential (one by one) |
| **What each model sees** | Different sample bootstrap | Full data, but weighted |
| **Primary goal** | Reduce **Variance** | Reduce **Bias** |
| **Use for** | High-variance models (deep trees) | High-bias models (shallow stumps) |
| **Risk** | Still inherits individual tree bias | Can overfit if too many trees |

---

## 2. The Bootstrap: Creating Diversity

### **What is Bootstrapping?**
Bootstrapping creates $B$ synthetic datasets by **sampling with replacement** from the original.

### **✍️ Worked Example**
Original dataset: 5 samples: `[A, B, C, D, E]`

Bootstrap Sample 1: `[A, C, A, E, B]` → OOB: `{D}`
Bootstrap Sample 2: `[B, B, D, A, C]` → OOB: `{E}`
Bootstrap Sample 3: `[E, A, A, D, B]` → OOB: `{C}`

- **Some samples appear multiple times** — the tree sees more emphasis on them.
- **Some samples never appear** — those are Out-of-Bag (OOB) — the built-in held-out set.

### **The 63.2% Rule (Derivation)**
Probability a specific sample is **never selected** across all $n$ draws:
$$P(\text{never selected}) = \left(1 - \frac{1}{n}\right)^n \xrightarrow{n \to \infty} \frac{1}{e} \approx 0.368$$

So **36.8% of samples are excluded** from any given bootstrap sample → **63.2% are included**. A mathematical constant, not a tunable parameter.

---

## 3. Random Forest: The Two Layers of Randomness

A plain bagging ensemble reduces variance but trees remain correlated (same dominant features). Random Forest adds **Feature Bagging** as a second layer.

### **Layer 1: Row Sampling (Bootstrap)**
Each tree is trained on a different bootstrap sample → different splits → moderate decorrelation.

### **Layer 2: Feature Sampling (Random Subspace Method)**
At **each split** within each tree, only $m$ randomly selected features are considered:
- Default: $m = \sqrt{F}$ for classification, $m = F/3$ for regression.
- Forces trees to use different features → much stronger decorrelation.

### **Why $\sqrt{F}$?**
- $m=1$: Trees too random → high bias. Individual accuracy collapses.
- $m=F$: Same as plain Bagging → high correlation $\rho$.
- $m=\sqrt{F}$: Empirically validated sweet spot (Breiman 2001) between tree diversity and individual accuracy.

### **Final Aggregation**
- **Classification:** Majority vote (or averaged probabilities).
- **Regression:** Mean across all $B$ tree predictions.

---

## 4. OOB Error: A Free Validation Set

**How OOB Error is Computed:**
1. For sample $i$, collect all trees that did **NOT** include sample $i$ in their bootstrap.
2. Run sample $i$ through ONLY those trees. Get predictions.
3. Aggregate → OOB prediction for sample $i$.
4. Repeat for all $n$ samples → compare to true labels.

**Why OOB ≈ Cross-Validation:** Each sample is tested on trees never trained on it — identical to $k$-fold CV logic. Research shows OOB error approximates leave-one-out CV error.

In Scikit-Learn: `oob_score=True` → access via `model.oob_score_`.

---

## 5. Feature Importance: MDI vs. Permutation

### **A. MDI — Mean Decrease in Impurity**
Default in Scikit-Learn (`model.feature_importances_`).

For each feature $f$: sum the Gini reduction weighted by sample count across all nodes across all trees where $f$ was used to split. Normalize so all sum to 1.

**Fatal Flaw:** Biased toward high-cardinality features — more unique values = more opportunities to accidentally reduce Gini. A random noise column can rank surprisingly high.

### **B. Permutation Importance (The Fix)**
1. Compute baseline score on test set.
2. Randomly shuffle feature $f$ in the test set (destroys its predictive signal).
3. Importance = Baseline Score − Shuffled Score.

Large drop → feature was truly important. Tiny drop → feature was irrelevant.

**Advantages:** Measured on test data, unbiased toward high-cardinality, captures actual predictive contribution.

In Scikit-Learn: `from sklearn.inspection import permutation_importance`

---

## 6. The Extrapolation Ceiling

Random Forest predictions are averages of leaf means from training data. The model **cannot predict beyond `[min(y_train), max(y_train)]`**.

**Example:** Train on stock prices $10–$500. In 2023 a stock reaches $1,200. The model physically cannot predict above $500 — it returns ~$490 regardless of inputs.

**Why Linear Regression can but trees can't:** $\hat{y} = 2x + 5$ can produce any value. Trees produce averages of training leaf values — bounded by training range.

**Impact on Time Series:** If future values exceed training range, trees systematically underestimate. Workaround: predict *differences* (returns) not absolute values.

---

## 7. Pros, Cons & When to Use

### **Pros**
1. **Hard to overfit:** Adding more trees only reduces variance. OOB error monotonically plateaus — never rises.
2. **Excellent default model:** Works well with minimal tuning (`n_estimators=100`, `max_features='sqrt'`).
3. **Native feature importances:** Built-in MDI signal for feature selection (supplement with permutation importance).
4. **Handles irrelevant features:** Feature sampling means irrelevant features are often simply not chosen.

### **Cons**
1. **Memory/Speed:** 500 deep trees require GB of RAM and slow inference.
2. **Not interpretable at ensemble level:** 500 paths for one sample. Need SHAP for explanation.
3. **Cannot extrapolate:** Bounded by training target range.
4. **Weak on high-dimensional sparse data:** Text/NLP tasks dominated by linear models.

### **Use When**
✅ Complex non-linear tabular data. ✅ Need a strong baseline quickly. ✅ Mixed feature types.

### **Don't Use When**
❌ Time series with out-of-range future values. ❌ Need legal/regulatory explainability (single tree is better). ❌ Real-time inference on edge devices.

---

## 8. Interpretability for Ensembles: SHAP Values

> **The problem:** In Chapter 1, we traced a fraud prediction through a single tree's path. That worked because one tree = one path. Now with 500 trees voting, there's no single path. Every tree disagrees. How do you explain the final prediction?

---

### **A. Why Ensemble Interpretability Is Hard**
A Random Forest with 500 trees classifies a transaction as fraud. Why?
- Tree 1: `amount > 5000 AND foreign = True` → fraud
- Tree 2: `time < 3am AND num_flags > 2` → fraud
- Tree 3: `foreign = True AND time < 6am` → fraud
- ...
- Tree 287: `amount > 8000` → legit
- Trees 1–499 vote: 312 fraud, 188 legit → **Prediction: Fraud**

There is no single "reason". 312 trees have 312 different paths. You cannot tell a manager "the model decided because of X" — it decided because of a *combination* of all trees. This is where SHAP comes in.

---

### **B. SHAP: SHapley Additive exPlanations**
SHAP is based on **Shapley values** from cooperative game theory (1953, Lloyd Shapley — Nobel Prize 2012). The core question SHAP asks is:

> *"If the features are 'players' in a game and the model's prediction is the 'payout', what is the fair contribution of each player (feature) to the total payout?"*

#### **The Core Idea (Marginal Contribution)**
The SHAP value for feature $f$ measures: *"On average, how much does including feature $f$ change the prediction, across all possible orderings of features?"*

$$\phi_f = \sum_{S \subseteq F \setminus \{f\}} \frac{|S|!(|F|-|S|-1)!}{|F|!} \left[v(S \cup \{f\}) - v(S)\right]$$

- $S$ — a subset of features **not including** $f$.
- $v(S)$ — the model's expected prediction using only features in $S$.
- $v(S \cup \{f\}) - v(S)$ — the *marginal contribution* of adding $f$ to subset $S$.
- The sum averages this marginal contribution over **all possible orderings** (all subsets $S$). This "fairness" across all orderings is what gives SHAP its theoretical guarantees.

#### **The Key Guarantee: Additivity**
$$\hat{y} = \text{base\_value} + \phi_1 + \phi_2 + \phi_3 + ... + \phi_F$$

- `base_value` = the model's average prediction across all training samples (the "baseline").
- $\phi_f$ = how much feature $f$'s actual value pushed the prediction **up or down** from the baseline.

This means SHAP values decompose the *difference* between the baseline and the prediction. They sum to **exactly** the model's output.

---

### **C. Worked Fraud Example with SHAP**

Random Forest predicts fraud with probability **0.87**. The base rate (average prediction across training) is **0.35**.

SHAP values for this transaction:

| Feature | Value | SHAP Value | Interpretation |
| :--- | :--- | :---: | :--- |
| `transaction_amount` | $8,000 | **+0.28** | High amount pushes prediction toward fraud by +28% |
| `is_foreign` | True | **+0.19** | Foreign origin adds another +19% |
| `time_of_day` | 2:30am | **+0.09** | Late hour adds +9% |
| `num_prior_flags` | 0 | **-0.04** | No prior flags slightly reduces fraud probability |

**Verification:** $0.35 + 0.28 + 0.19 + 0.09 - 0.04 = 0.87$ ✓

**The explanation in plain English:**
> *"This transaction was classified as fraud (87% probability). Compared to the average transaction (35% fraud rate), the primary drivers pushing this higher were: the large amount of $8,000 (+28%), the foreign country origin (+19%), and the early morning timing (+9%). The only factor reducing suspicion was the clean prior history (-4%)."

This is an ensemble explanation. No single tree's path — but a fair, additive decomposition across all 500 trees.

---

### **D. TreeSHAP: Fast SHAP for Tree-Based Models**
Computing exact Shapley values naively requires $2^F$ model evaluations (exponential in features). For a general model with hundreds of features, this is infeasible.

**TreeSHAP** (Lundberg et al., 2018) is an algorithm specifically designed for tree-based models that computes exact Shapley values in **$O(TLD^2)$ time** — where $T$ = number of trees, $L$ = max leaves per tree, $D$ = max depth. It exploits the tree structure to avoid the exponential blowup.

It works for:
- `sklearn` Decision Trees and Random Forests
- XGBoost
- LightGBM
- CatBoost

---

### **E. Types of SHAP Plots and What They Tell You**

#### **1. Waterfall Plot — One Sample Explanation**
```python
import shap
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.plots.waterfall(explainer(X_test)[0])   # explain first sample
```
Shows base value → each feature's contribution → final prediction as a horizontal bar chart. Red = pushes toward positive class. Blue = pushes away. Ideal for explaining ONE prediction to a stakeholder.

#### **2. Beeswarm / Summary Plot — Global Feature Importance**
```python
shap.summary_plot(shap_values, X_test, feature_names=feature_names)
```
One row per feature; dots = individual samples. Color = feature value (red = high, blue = low). X-axis = SHAP value. Shows BOTH global importance AND the direction of effect. Far superior to MDI or permutation importance because it shows:
- Which features matter most overall.
- Whether high values of a feature push predictions up or down.
- Non-monotonic relationships (some features push up at low values and down at high values).

#### **3. Dependence Plot — Feature Interaction**
```python
shap.dependence_plot('transaction_amount', shap_values, X_test)
```
X-axis = feature value. Y-axis = SHAP value for that feature. Color = another feature (auto-selected by highest interaction). Reveals: is the effect linear? Is there a threshold? Does a third feature change the relationship?

#### **4. Force Plot — One Sample, Interactive**
```python
shap.force_plot(explainer.expected_value, shap_values[0], X_test.iloc[0])
```
An interactive horizontal waterfall showing features pushing left (reducing prediction) or right (increasing prediction).

---

### **F. SHAP vs. Other Interpretability Methods**

| Method | Scope | Applies to | Advantage | Limitation |
| :--- | :--- | :--- | :--- | :--- |
| **Decision Path** | Local (1 sample) | Single Tree only | Exact structural trace | Breaks for ensembles |
| **MDI Importance** | Global (all samples) | Trees | Fast, built-in | Biased toward high-cardinality |
| **Permutation Importance** | Global | Any model | Unbiased, test-set based | Slow, ignores feature interactions |
| **SHAP** | Both local & global | Any model (TreeSHAP for trees) | Theoretically grounded, additive, consistent | Requires `shap` library |
| **LIME** | Local (1 sample) | Any model | Model-agnostic | Approximation — not exact |

**SHAP is the industry standard** for production ML model explanation, especially for:
- Financial services (loan denial explanation, fraud alerts).
- Healthcare (treatment recommendation audit).
- Any model subject to fairness/bias audits.

---

### **G. Key Properties That Make SHAP Trustworthy**

1. **Efficiency:** $\sum_f \phi_f = \hat{y} - E[\hat{y}]$. The SHAP values always sum to the exact model output. No information is lost.
2. **Symmetry:** If two features contribute identically to every prediction, their SHAP values are equal. No arbitrary ordering bias.
3. **Dummy Feature Property:** If a feature has zero marginal contribution to every prediction, its SHAP value is exactly 0. Irrelevant features get score 0 — not noise.
4. **Consistency:** If a model changes so that feature A's contribution increases, A's SHAP value can only increase (or stay the same). The ranking is stable under model changes.

These four axioms are what distinguish SHAP from heuristic methods like MDI or gradient-based importance.

---

## 9. Interview Deep Dive (12 Questions)

### **Q1: Why does averaging multiple trees reduce variance? Prove it.**
**A:** For $N$ independent trees each with variance $\sigma^2$: $Var(\bar{X}) = \frac{1}{N^2}\sum Var(X_i) = \frac{N\sigma^2}{N^2} = \frac{\sigma^2}{N}$. Bias doesn't change — average of unbiased estimators is unbiased. So we get 100x variance reduction (for 100 trees) at no cost to bias.

### **Q2: Why does RF need feature bagging on top of row bagging?**
**A:** Row sampling creates different datasets but trees still use the same dominant features → high pairwise correlation $\rho$. The ensemble variance floor is $\rho\sigma^2$ — cannot be reduced by more trees. Feature bagging forces some trees to build without access to the dominant feature, creating structurally different trees with lower $\rho$. This drives down both $\rho\sigma^2$ (the floor) and $\frac{(1-\rho)}{N}\sigma^2$.

### **Q3: Prove why ~36.8% of samples are OOB.**
**A:** Probability a specific sample is not drawn in one draw = $(1-1/n)$. Across $n$ draws: $(1-1/n)^n$. As $n\to\infty$: $1/e \approx 0.368$. So 36.8% are always OOB.

### **Q4: How is OOB error computed across the full forest?**
**A:** For each training sample $i$, collect trees that did NOT include $i$ in their bootstrap. Run $i$ through only those trees. Aggregate predictions. Compare to true label. Repeat for all $n$ samples. This approximates leave-one-out cross-validation error.

### **Q5: What is MDI and its flaw? How do you fix it?**
**A:** MDI = total weighted Gini reduction per feature across all nodes in all trees. Flaw: biased toward high-cardinality features (more split opportunities = higher score by chance). Fix: Permutation Importance (shuffle feature in test set, measure score drop). Or SHAP values (theoretically grounded, unbiased).

### **Q6: Why doesn't adding more trees to RF hurt accuracy?**
**A:** Trees are built independently. Each additional tree adds one more independent vote. Ensemble variance $\sigma^2/N$ can only decrease or plateau — never increase. This contrasts with Boosting where each new tree changes the entire model's structure.

### **Q7: What is the extrapolation limitation of Random Forest?**
**A:** RF predictions are weighted averages of leaf means, which are bounded by the training target range `[min(y_train), max(y_train)]`. A Linear Regression $\hat{y} = wx + b$ can extrapolate to any value. Trees cannot. This makes RF unsuitable for forecasting when future values may exceed the training range.

### **Q8: What are SHAP values and why are they better than MDI for explaining predictions?**
**A:** SHAP values are based on Shapley values from game theory. They measure the average marginal contribution of each feature across all possible subsets of features. MDI only measures global importance during training (biased). SHAP:  (1) explains INDIVIDUAL predictions (local scope), (2) is theoretically guaranteed to sum to the exact model output, (3) is unbiased toward high-cardinality features, (4) shows both magnitude AND direction of each feature's effect.

### **Q9: Explain the SHAP additivity guarantee with a concrete example.**
**A:** If the model's average prediction across training is 0.35 (base value), and for a specific sample the SHAP values are +0.28 (amount), +0.19 (foreign), +0.09 (time), -0.04 (no prior flags), then: $0.35 + 0.28 + 0.19 + 0.09 - 0.04 = 0.87$. The model predicted 0.87 fraud probability. The SHAP values sum to **exactly** the gap between the baseline (0.35) and the prediction (0.87). No approximation, no information lost.

### **Q10: What is TreeSHAP and why is it important?**
**A:** Computing Shapley values naively requires $2^F$ model evaluations (exponential). TreeSHAP exploits tree structure for $O(TLD^2)$ exact computation — polynomial in tree depth, not exponential in features. Makes SHAP feasible for production Random Forests, XGBoost, and LightGBM with hundreds of features.

### **Q11: How would you use SHAP to explain a fraud flagging to a customer?**
**A:** (1) Compute SHAP values for the flagged transaction. (2) Identify the top 3-4 features with the largest absolute SHAP values. (3) Translate to plain language: "Your transaction was flagged primarily because [feature with highest positive SHAP] was unusual compared to typical transactions at our bank. Additionally, [second feature] contributed. The only factor in your favor was [negative SHAP feature]." This complies with GDPR's right-to-explanation requirement.

### **Q12: Bagging reduces variance. Does it also reduce bias?**
**A:** No. $E[\bar{X}] = E[X]$ — averaging doesn't change the expectation. If each tree has a systematic bias (consistently underestimates for high-income samples), the average also underestimates. Bagging works best with low-bias, high-variance learners (deep unconstrained trees). Applying bagging to a biased shallow tree doesn't help the bias.

---

## 10. Chapter Summary & Interview Checklist

| # | Question | Minimum Expected Answer |
| :- | :--- | :--- |
| 1 | Why does averaging models reduce variance? | $Var(\bar{X}) = \sigma^2/N$ for independent models. |
| 2 | Role of correlation $\rho$ in ensemble variance? | High $\rho$ → floor $\rho\sigma^2$ can't be reduced by more trees. |
| 3 | Why is 36.8% OOB? | $(1-1/n)^n \to 1/e$. Mathematical constant. |
| 4 | Two layers of randomness in RF? | Row bagging + Feature bagging. |
| 5 | Default `max_features` and why? | $\sqrt{F}$ — Breiman's empirically validated sweet spot. |
| 6 | How is OOB error computed? | Run sample through its OOB trees only. Aggregate. |
| 7 | MDI flaw and fix? | Biased toward high-cardinality. Fix: Permutation Importance or SHAP. |
| 8 | Why can't RF extrapolate? | Predictions bounded by training leaf means. |
| 9 | What is TreeSHAP? | Fast exact Shapley values for trees in $O(TLD^2)$. |
| 10 | SHAP additivity guarantee? | $\hat{y} = base\_value + \sum \phi_f$. Exact decomposition. |
| 11 | SHAP vs MDI vs Permutation Importance? | SHAP: local+global, exact, theoretically grounded. MDI: biased. Perm: test-set based but global only. |
| 12 | Does Bagging reduce bias? | No. Only reduces variance. |

---

## 11. Implementation Playground (Placeholder)

Five cells — each targeting a specific concept from this chapter.


In [ ]:
# ─── CELL 1: Single Tree vs. Random Forest — Variance Demonstration ────────────
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
import numpy as np

# TODO: Create a synthetic dataset
pass

# TODO: Run 10-fold CV for a deep single tree (max_depth=None)
# Observe the SPREAD of scores (std deviation = variance in performance)
pass

# TODO: Run 10-fold CV for a Random Forest (n_estimators=100)
# Compare: RF mean should be higher AND std should be lower
pass

In [ ]:
# ─── CELL 2: n_estimators Convergence Curve ────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# TODO: Create dataset, split train/test
pass

# TODO: Train RF with oob_score=True for n_estimators in [1, 5, 10, 25, 50, 100, 200, 500]
# Record oob_score_ and test score at each
pass

# TODO: Plot n_estimators vs OOB Score AND Test Score
# Observe: both converge and plateau. OOB ≈ Test. Score never decreases.
pass

In [ ]:
# ─── CELL 3: OOB Score vs Cross-Validation Score Comparison ────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
import numpy as np

# TODO: Load Breast Cancer dataset
pass

# TODO: Train RF with oob_score=True, n_estimators=200. Print oob_score_.
pass

# TODO: Run 5-fold cross_val_score. Compare mean CV Score vs OOB Score.
# They should be very close — verifies OOB ≈ CV
pass

In [ ]:
# ─── CELL 4: MDI vs Permutation Importance Comparison ─────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

# TODO: Load data, split, train RF
pass

# TODO: Get MDI importance from model.feature_importances_
pass

# TODO: Get Permutation Importance on TEST set
pass

# TODO: Side-by-side bar chart comparing MDI vs Permutation
# BONUS: Add a random noise column before training
# Observe: MDI ranks it high. Permutation ranks it near zero.
pass

In [ ]:
# ─── CELL 5: SHAP Explanation for Fraud Detection ─────────────────────────────
# pip install shap  (if not already installed)
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

feature_names = ['transaction_amount', 'is_foreign', 'time_of_day', 'num_prior_flags']

# TODO: Create a synthetic fraud dataset with the 4 features above, binary target
pass

# TODO: Train a RandomForestClassifier
pass

# ── PART A: TreeSHAP Explainer ──────────────────────────────────────────────────
# TODO: explainer = shap.TreeExplainer(model)
# TODO: shap_values = explainer.shap_values(X_test)
# Note: for binary classification, shap_values is a list of 2 arrays [class_0, class_1]
# Use shap_values[1] for the fraud (positive) class
pass

# ── PART B: Explain ONE fraud prediction ───────────────────────────────────────
# TODO: Pick the first test sample classified as fraud
# TODO: Print: base_value, each feature's shap value, and verify they sum to prediction
# Expected: base_value + sum(shap_values) == model.predict_proba(sample)[1]
pass

# ── PART C: Waterfall Plot ──────────────────────────────────────────────────────
# TODO: shap.plots.waterfall(explainer(X_test)[sample_idx])
# Read: which features pushed toward fraud (red) vs away (blue)?
pass

# ── PART D: Global Summary (Beeswarm) Plot ─────────────────────────────────────
# TODO: shap.summary_plot(shap_values[1], X_test, feature_names=feature_names)
# Read: which features have the highest AVERAGE impact across all test samples?
# Compare this ranking to MDI from Cell 4 — are they the same or different?
pass